# Capítulo 14: Segmentación Inteligente de Clientes
## Notebook de Práctica — Clustering con K-Means y Naming con IA Generativa

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('Librerías cargadas correctamente.')

## 1. Carga y Exploración de Datos

Cargamos el dataset de 3000 clientes de e-commerce y exploramos sus variables.

In [ ]:
df = pd.read_csv('../datos/datos_ecommerce_clientes.csv')
print(f'Dimensiones: {df.shape}')
print(f'\nPrimeras filas:')
df.head(10)

In [ ]:
df.describe()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

variables = ['total_purchases', 'avg_order_value', 'days_since_last_purchase',
             'email_engagement_score', 'support_tickets_count', 'lifetime_value']

for ax, var in zip(axes.flatten(), variables):
    ax.hist(df[var], bins=40, edgecolor='black', alpha=0.7, color='steelblue')
    ax.set_title(var, fontsize=12)
    ax.set_xlabel('')

plt.suptitle('Distribución de Variables Numéricas', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 2. Preprocesamiento

Seleccionamos las variables numéricas relevantes para el clustering y las estandarizamos.

In [ ]:
features = ['total_purchases', 'avg_order_value', 'days_since_last_purchase',
            'email_engagement_score', 'support_tickets_count', 'lifetime_value']

X = df[features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=features)

print('Datos estandarizados. Media ≈ 0, Desviación ≈ 1.')
X_scaled.describe().round(2)

## 3. Método del Codo (Elbow Method)

Buscamos el número óptimo de clusters evaluando la inercia para diferentes valores de k.

In [ ]:
inertias = []
sil_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    inertias.append(kmeans.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(K_range, inertias, 'bo-', linewidth=2)
ax1.set_xlabel('Número de Clusters (k)')
ax1.set_ylabel('Inercia')
ax1.set_title('Método del Codo')

ax2.plot(K_range, sil_scores, 'rs-', linewidth=2)
ax2.set_xlabel('Número de Clusters (k)')
ax2.set_ylabel('Puntuación de Silueta')
ax2.set_title('Puntuación de Silueta por k')

plt.tight_layout()
plt.show()

best_k = K_range[np.argmax(sil_scores)]
print(f'\nMejor k por silueta: {best_k} (score: {max(sil_scores):.3f})')

## 4. K-Means con k óptimo

Entrenamos el modelo con el número de clusters óptimo y analizamos la silueta de cada punto.

In [ ]:
kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['cluster'] = kmeans_final.fit_predict(X_scaled)

sil_global = silhouette_score(X_scaled, df['cluster'])
print(f'Silueta global: {sil_global:.3f}')
print(f'\nDistribución de clientes por cluster:')
print(df['cluster'].value_counts().sort_index())

In [ ]:
from sklearn.metrics import silhouette_samples

sample_silhouette = silhouette_samples(X_scaled, df['cluster'])
df['silhouette'] = sample_silhouette

fig, ax = plt.subplots(figsize=(10, 6))
y_lower = 10

for i in range(best_k):
    cluster_sil = sample_silhouette[df['cluster'] == i]
    cluster_sil.sort()
    size = len(cluster_sil)
    y_upper = y_lower + size
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_sil, alpha=0.7, label=f'Cluster {i}')
    y_lower = y_upper + 10

ax.axvline(x=sil_global, color='red', linestyle='--', label=f'Media: {sil_global:.3f}')
ax.set_title('Gráfico de Silueta por Cluster')
ax.set_xlabel('Coeficiente de Silueta')
ax.set_ylabel('Clientes (ordenados por cluster)')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Perfiles de Cluster — Interpretación

Analizamos las características promedio de cada cluster para entender qué los define.

In [ ]:
profiles = df.groupby('cluster')[features].mean().round(2)
profiles['n_clientes'] = df['cluster'].value_counts().sort_index()
print('Perfiles promedio por cluster:')
profiles

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, var in zip(axes.flatten(), features):
    df.boxplot(column=var, by='cluster', ax=ax)
    ax.set_title(var)
    ax.set_xlabel('Cluster')
    ax.set_ylabel('')

plt.suptitle('Distribución por Cluster', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 6. Naming con IA Generativa

Usamos un LLM para generar nombres descriptivos basados en los perfiles cuantitativos.

In [ ]:
profiles_dict = profiles.drop(columns='n_clientes').to_dict(orient='index')

print('Prompt para IA generativa (copiar y pegar en ChatGPT/Claude):\n')
print('='*70)
for cluster_id, metrics in profiles_dict.items():
    print(f'\nCluster {cluster_id}:')
    for metric, value in metrics.items():
        print(f'  - {metric}: {value}')

print('\n' + '='*70)
print('Sugerencia de prompt:\n')
print('''"Basándote en estos perfiles de clientes de e-commerce, genera un nombre 
evocador y un eslogan corto para cada cluster. Sé creativo pero preciso. 
Formato: Cluster X: Nombre — Eslogan"''')

### Ejemplo de nombres generados por IA:

| Cluster | Nombre | Eslogan |
|---------|--------|---------|
| 0 | **Los Recurrentes** | "Volver siempre es su costumbre" |
| 1 | **Los Dorados** | "Calidad sobre cantidad" |
| 2 | **Los Adormilados** | "Se fueron y no volvieron" |
| 3 | **Los Exploradores** | "Siempre buscando lo nuevo" |
| 4 | **Los Eficientes** | "Compran poco pero acertado" |

## 7. Análisis por Región y Categoría

Exploramos cómo se distribuyen los clusters por región y preferencia de categoría.

In [ ]:
ct_region = pd.crosstab(df['region'], df['cluster'], normalize='index').round(3)
ct_region.plot(kind='bar', stacked=True, figsize=(12, 6), colormap='viridis')
plt.title('Distribución de Clusters por Región')
plt.ylabel('Proporción')
plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()

In [ ]:
ct_cat = pd.crosstab(df['product_category_preference'], df['cluster'], normalize='index').round(3)
ct_cat.plot(kind='bar', stacked=True, figsize=(12, 6), colormap='Set2')
plt.title('Distribución de Clusters por Categoría de Producto')
plt.ylabel('Proporción')
plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()

## 8. Consideraciones Éticas

**¿Los segmentos discriminan?**

La segmentación basada en datos de comportamiento puede:
- **Discriminación indirecta**: Si la región o el ingreso (proxy del lifetime_value) correlaciona con variables protegidas (etnia, género), los segmentos pueden perpetuar sesgos.
- **Exclusión de servicios**: Segmentos con bajo engagement podrían recibir menos atención o peores ofertas.
- **Perfilamiento excesivo**: Usar datos de emails y tickets para perfilar puede invadir la privacidad.

**Recomendaciones éticas:**
1. Auditar si los clusters correlacionan con variables demográficas protegidas.
2. No usar segmentos para negar servicios o precios diferenciados injustamente.
3. Dar a los clientes derecho a saber en qué segmento están.
4. Revisar periódicamente si los modelos perpetúan sesgos históricos.

## 9. Referencias

- Kaufman, L., & Rousseeuw, P. J. (2005). *Finding Groups in Data: An Introduction to Cluster Analysis*. Wiley.
- Ngai, E. W. T. (2009). Customer relationship management research (1992–2002). *Marketing Intelligence & Planning*, 27(2), 220–246.
- Van der Laan, M. J., & Pollard, K. S. (2003). A new algorithm for hybrid hierarchical clustering. *Journal of Statistical Computation and Simulation*, 73(8), 555–568.